In [1]:
from scipy.stats import kde
import h5py
import astropy.io.fits as fits
import csv
import pandas as pd
import numpy as np
import healpy as hp
import tables
import pickle
import os
from astropy.table import Table
from astropy.coordinates import SkyCoord
from tqdm import tqdm
from astropy.io import ascii
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import incredible as cr
from scipy.special import erf
from scipy import stats
import scipy.optimize as opt
from scipy import stats
import scipy.optimize as opt
import emcee
import tqdm
import pickle
from astropy import table
from astropy.table import Table, join, unique
from specutils import SpectralRegion
from scipy.interpolate import BSpline, make_interp_spline, UnivariateSpline

from astropy.cosmology import FlatLambdaCDM
from scipy.interpolate import interp1d
import numpy as np
import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18
from astropy.cosmology import z_at_value
cosmo = Planck18

## Functions for computing spectroscopic richness and profiles
from projection_functions import *


##Plotting
import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt

from astropy.io import ascii

plot_params = {

'axes.linewidth': 1.25,
'xtick.major.width': 1, 'ytick.major.width': 1,
'xtick.minor.width': 1, 'ytick.minor.width': 1,
'xtick.major.size': 12, 'ytick.major.size': 12,
'xtick.labelsize': 12, 'ytick.labelsize': 12,
'xtick.direction': 'in', 'ytick.direction': 'in',
'xtick.major.pad': 6, 'xtick.minor.pad': 6,
'figure.constrained_layout.use': True,
#'text.usetex': True,
'font.family': 'Serif',
'font.size': 14,
'legend.fontsize': 11
    
}
plt.rcParams.update(plot_params)

In [2]:
import astropy.units as u
from astropy.coordinates import SkyCoord
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18

## Importing the redMaPPer Catalog with matched BGS galaxies

In [3]:
with open('bgs_clus_RM_gal_matched.pickle', 'rb') as handle:
    bgs_matched = pickle.load(handle)

fcols = ['g','r','z','w1','w2']
for col in fcols:
    #bgs_matched['flux_'+col.lower()+'_dered'] = bgs_matched['FLUX_'+col]/data['MW_TRANSMISSION_'+col]
    bgs_matched['r_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_r_dered'])
    bgs_matched['g_dered'] = 22.5 - 2.5*np.log10(bgs_matched['flux_g_dered'])
    bgs_matched['gmr'] = bgs_matched['g_dered']-bgs_matched['r_dered']

#binGap = 1e-5
wide_bin_1 = np.linspace(-0.1,-0.005,21, endpoint=False)
small_bin = np.linspace(-0.005,0.005,21, endpoint=False)
micro_bin = np.linspace(-0.005,0.005,31)
wide_bin_2 = np.linspace(0.005,0.1,21, endpoint=False)

binBoundaries = np.hstack((wide_bin_1, small_bin))
binBoundaries = np.hstack((binBoundaries, wide_bin_2))
binCent = np.asarray([(binBoundaries[i] + binBoundaries[i+1])/2 for i in range(len(binBoundaries)-1)])
binCent_micro = np.asarray([(micro_bin[i] + micro_bin[i+1])/2 for i in range(len(micro_bin)-1)])
bin_width = np.asarray([binBoundaries[i+1]-binBoundaries[i] for i in range(len(binBoundaries[:-1]))])

## None overlapping bins
assert len(set(binCent)) == len(binCent), "Overlapping bins"
assert len(set(binBoundaries)) == len(binBoundaries), "Overlapping bins"

In [4]:
rm_clus = unique(bgs_matched, keys='ID')[::1]

In [9]:
##Save rm_clus as a pickle file
with open("RMClusters_matched.pkl", "wb") as f:
    pickle.dump(rm_clus, f)


## Catalog matching function

In [7]:
import healpy as hp
import numpy as np
from collections import defaultdict

NSIDE = 2048

def radec_to_pix(ra, dec, nside=NSIDE):
    theta = np.deg2rad(90 - dec)
    phi   = np.deg2rad(ra)
    return hp.ang2pix(nside, theta, phi)

def neighbor_pixels(ra, dec, radius, nside=NSIDE):
    vec = hp.ang2vec(ra, dec, lonlat=True)

    return hp.query_disc(nside, vec, radius, inclusive=True)

def angsep(ra1, dec1, ra2, dec2):
    ra1  = np.deg2rad(ra1)
    dec1 = np.deg2rad(dec1)
    ra2  = np.deg2rad(ra2)
    dec2 = np.deg2rad(dec2)

    return np.arccos(
        np.sin(dec1)*np.sin(dec2) + 
        np.cos(dec1)*np.cos(dec2)*np.cos(ra1 - ra2)
    )   # returns radians

In [8]:
pix_gal = radec_to_pix(SDSS_gal['ra'], SDSS_gal['dec'], NSIDE)

NameError: name 'SDSS_gal' is not defined

In [ ]:
gal_index = defaultdict(list)
for i, p in enumerate(pix_gal):
    gal_id = SDSS_gal[i]['objid']
    gal_index[p].append([i, gal_id])

In [ ]:
from astropy.cosmology import Planck18 as cosmo

R_match = 10.0  # Mpc

cluster_matches = []

for i in range(len(rm_clus)):
    ra_c  = rm_clus[i]['RA_x']
    dec_c = rm_clus[i]['DEC_x']
    z_c   = rm_clus[i]['Z_SPEC_x']
    id_c = rm_clus[i]['ID']

    # projected comoving distance threshold
    D_M   = cosmo.comoving_transverse_distance(z_c).value  # Mpc
    theta_max_rad = R_match / D_M                          # radians

    # neighbor pixels
    neigh_pix = neighbor_pixels(ra_c, dec_c, theta_max_rad, NSIDE)

    # gather candidate galaxy indices

    cand = []
    for p in neigh_pix:
        cand.extend(gal_index.get(p, []))
    if not cand:
        continue

    cand = np.array(cand, dtype=np.int64)
    cand_ind = cand[:,0]
    cand_ID = cand[:,1]


    # angular separation
    sep = angsep(ra_c, dec_c, SDSS_gal[cand_ind]['ra'], SDSS_gal[cand_ind]['dec'])   # radians

    # compute projected comoving separation R_perp = D_M * theta
    Rperp = D_M * sep                                        # Mpc

    # mask for R_perp < 8 Mpc
    keep = Rperp < R_match

    # store matches
    for g in cand[keep]:
        ind_g =  g[0]
        id_g = g[1] 
        cluster_matches.append([i,ind_g])

cluster_matches = np.array(cluster_matches)

In [13]:
from astropy.cosmology import Planck18 as cosmo

class ClusterCatalog:
    def __init__(self, path, cosmology):
        self.path = path
        self.data = None  # not loaded yet
        self.cosmo = cosmology

    def load(self):
        if self.data is None:
            with open(self.path, 'rb') as handle:
                rm_clus = pickle.load(handle)

        self.data = rm_clus

        ##Create an environment galaxy count if none. 
        if "Ngal_env" not in self.data.colnames:
            self.data["Ngal_env"] = np.zeros(len(self.data))

    '''
    Allow this function to iterate through multiple galaxy files. 
    '''
    def compute_galCount_localEnv(galCatalog, R=8, nside=2048):
        Ngal = self._matchCatalog(galCatalog, R_env, nside)

    
    def _matchCatalog(galCatalog, nside):
        cluster_matches = []

        pix_gal = radec_to_pix(galCatalog['ra'], galCatalog['dec'], nside)
        gal_index = defaultdict(list)
        for i, p in enumerate(pix_gal):
            gal_id = galCatalog[i]['objid']
            gal_index[p].append([i, gal_id])

        for i in range(len(self.data)):
            ra_c  = self.data[i]['RA_x'] #_x refers to the properties of the BCG
            dec_c = self.data[i]['DEC_x']
            z_c   = self.data[i]['Z_SPEC_x']
            id_c = self.data[i]['ID']
        
            # projected comoving distance threshold
            D_M   = self.cosmo.comoving_transverse_distance(z_c).value  # Mpc
            theta_max_rad = R_match / D_M                          # radians
        
            # neighbor pixels
            neigh_pix = neighbor_pixels(ra_c, dec_c, theta_max_rad, nside)
        
            # gather candidate galaxy indices
            cand = []
            for p in neigh_pix:
                cand.extend(gal_index.get(p, []))
            if not cand:
                continue
        
            cand = np.array(cand, dtype=np.int64)
            cand_ind = cand[:,0]
            cand_ID = cand[:,1]
                
            # angular separation
            sep = angsep(ra_c, dec_c, SDSS_gal[cand_ind]['ra'], SDSS_gal[cand_ind]['dec'])   # radians
        
            # compute projected comoving separation R_perp = D_M * theta
            Rperp = D_M * sep                                        # Mpc
        
            # mask for R_perp < 8 Mpc
            keep = Rperp < R_match
        
            # store matches
            for g in cand[keep]:
                ind_g =  g[0]
                id_g = g[1] 
                cluster_matches.append([i,ind_g])
        
        cluster_matches = np.array(cluster_matches)
        

In [16]:
## Testing the pipeline

In [14]:
test = ClusterCatalog('RMClusters_matched.pkl', cosmo)

In [15]:
test.load()